# Spark Performance Lab — Broadcast, Skew, Partitions, AQE

This lab is a hands-on Spark performance notebook built around a Citi-style telemetry story.

## Mental model

Spark performance usually comes down to four big levers:

1. **Move less data**  
   Use predicate pushdown and projection so Spark reads less from the source.

2. **Shuffle less data**  
   Broadcast small dimensions when possible to avoid expensive wide shuffles.

3. **Balance the work**  
   Avoid partition skew so one task does not become the bottleneck for the whole stage.

4. **Use the right partition shape**  
   Too few partitions underutilize the cluster. Too many partitions create overhead. The sweet spot depends on data size and operation type.

## Dataset context

- **PostgreSQL**: `localhost:5432`, database `de_telemetry`
- **User**: `de_admin`
- **Password**: `DeAdmin2026!`

Tables:

- `endpoints`: 10,000 rows  
  `endpoint_id (int PK), name (varchar), region (varchar), status (varchar), category (varchar)`

- `metrics`: 500,000 rows  
  `endpoint_id (int FK), metric_name (varchar), value (float), timestamp (timestamptz)`

- `alerts`: 25,000 rows  
  `alert_id (int PK), endpoint_id (int FK), severity (varchar), message (text), created_at (timestamptz)`

Narrative:

> Citi monitors 6,000+ API endpoints for latency, error rate, and throughput. Alerts escalate through severity tiers. At this scale, small Spark tuning decisions materially change runtime.

## What you will measure

1. Broadcast join
2. Shuffle partition tuning
3. Skew simulation and salting
4. `coalesce()` vs `repartition()`
5. Predicate pushdown via JDBC

## Notes

- This notebook uses **PySpark 3.5.4**
- Spark master is `local[*]`
- AQE is enabled so you can observe adaptive behavior
- No package install cells are included


In [ ]:
import os
import time
from contextlib import contextmanager

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window


POSTGRES_HOST = "localhost"
POSTGRES_PORT = 5432
POSTGRES_DB = "de_telemetry"
POSTGRES_USER = "de_admin"
POSTGRES_PASSWORD = "DeAdmin2026!"
POSTGRES_URL = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

JDBC_PROPERTIES = {
    "user": POSTGRES_USER,
    "password": POSTGRES_PASSWORD,
    "driver": "org.postgresql.Driver",
}

APP_NAME = "spark_performance_lab"

def build_spark():
    existing = SparkSession.getActiveSession()
    if existing is not None:
        existing.stop()

    builder = (
        SparkSession.builder
        .appName(APP_NAME)
        .master("local[*]")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.adaptive.skewJoin.enabled", "true")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.showConsoleProgress", "true")
        .config("spark.sql.broadcastTimeout", "1200")
        .config("spark.driver.extraJavaOptions", "-Duser.timezone=UTC")
        .config("spark.executor.extraJavaOptions", "-Duser.timezone=UTC")
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
    )
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark

spark = build_spark()

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions", "200 (Spark default)"))

@contextmanager
def timer(label):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.4f}s")

def timed_call(label, fn):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.4f}s")
    return result, elapsed

def read_jdbc_table(table_or_query, num_partitions=None, partition_column=None, lower_bound=None, upper_bound=None, fetchsize=10000):
    reader = (
        spark.read.format("jdbc")
        .option("url", POSTGRES_URL)
        .option("dbtable", table_or_query)
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .option("fetchsize", str(fetchsize))
    )
    if num_partitions is not None and partition_column is not None and lower_bound is not None and upper_bound is not None:
        reader = (
            reader.option("numPartitions", str(num_partitions))
            .option("partitionColumn", partition_column)
            .option("lowerBound", str(lower_bound))
            .option("upperBound", str(upper_bound))
        )
    return reader.load()

endpoints_df = read_jdbc_table("public.endpoints", num_partitions=8, partition_column="endpoint_id", lower_bound=1, upper_bound=10000).cache()
alerts_df = read_jdbc_table("public.alerts", num_partitions=8, partition_column="alert_id", lower_bound=1, upper_bound=25000).cache()
metrics_df = read_jdbc_table("(select endpoint_id, metric_name, value, timestamp from public.metrics) as metrics_base", num_partitions=16, partition_column="endpoint_id", lower_bound=1, upper_bound=10000).cache()

# Materialize caches once so timing in later cells reflects experiment work, not first-load variability.
_ = endpoints_df.count()
_ = alerts_df.count()
_ = metrics_df.count()

print("endpoints rows:", endpoints_df.count())
print("alerts rows:", alerts_df.count())
print("metrics rows:", metrics_df.count())


## Experiment 1 — Broadcast Join

Goal: join `alerts` (25K) to `endpoints` (10K) with and without a broadcast hint.

Why this matters:

- `endpoints` is a small dimension table
- broadcasting it can eliminate a shuffle-heavy sort merge join
- that often changes a "move both sides across the network" plan into a "ship the small side once" plan

We will:

1. Run the join without a broadcast hint
2. Run the join with `broadcast(endpoints_df)`
3. Force execution with `.count()`
4. Compare elapsed times and print a speedup


In [ ]:
spark.catalog.clearCache()

join_select = [
    alerts_df["alert_id"],
    alerts_df["endpoint_id"],
    alerts_df["severity"],
    alerts_df["created_at"],
    endpoints_df["name"].alias("endpoint_name"),
    endpoints_df["region"],
    endpoints_df["status"],
    endpoints_df["category"],
]

def run_non_broadcast_join():
    joined = alerts_df.join(endpoints_df, on="endpoint_id", how="inner").select(*join_select)
    return joined.count()

def run_broadcast_join():
    joined = alerts_df.join(F.broadcast(endpoints_df), on="endpoint_id", how="inner").select(*join_select)
    return joined.count()

_, no_broadcast_time = timed_call("Without broadcast", run_non_broadcast_join)
_, with_broadcast_time = timed_call("With broadcast", run_broadcast_join)

speedup = (no_broadcast_time / with_broadcast_time) if with_broadcast_time > 0 else float("inf")
print(f"Without broadcast: {no_broadcast_time:.4f}s, With broadcast: {with_broadcast_time:.4f}s, Speedup: {speedup:.2f}x")

print("\nExecution plan without broadcast:")
alerts_df.join(endpoints_df, on="endpoint_id", how="inner").explain(mode="formatted")

print("\nExecution plan with broadcast:")
alerts_df.join(F.broadcast(endpoints_df), on="endpoint_id", how="inner").explain(mode="formatted")


## Experiment 2 — Partition Tuning

Goal: test `spark.sql.shuffle.partitions` with values `2, 10, 50, 200`.

Why this matters:

- **Too few partitions**: large tasks, weak parallelism
- **Too many partitions**: scheduler overhead, many tiny tasks
- The best value is usually a **sweet spot**, not the minimum or maximum

Workload:

- group `alerts` by severity
- include a join to endpoints so we generate a realistic shuffle pattern
- time the action for each partition setting


In [ ]:
partition_results = []

for shuffle_partitions in [2, 10, 50, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_partitions))

    def run_partition_test():
        grouped = (
            alerts_df.join(F.broadcast(endpoints_df.select("endpoint_id", "region")), on="endpoint_id", how="inner")
            .groupBy("severity", "region")
            .agg(F.count("*").alias("alert_count"))
            .orderBy("severity", "region")
        )
        return grouped.collect()

    rows, elapsed = timed_call(f"shuffle_partitions={shuffle_partitions}", run_partition_test)
    partition_results.append((shuffle_partitions, elapsed, len(rows)))

results_df = spark.createDataFrame(partition_results, ["shuffle_partitions", "elapsed_seconds", "result_rows"]).orderBy("shuffle_partitions")
results_df.show(truncate=False)

best_row = results_df.orderBy("elapsed_seconds").first()
print(
    f"Sweet spot in this run: {best_row['shuffle_partitions']} partitions "
    f"at {best_row['elapsed_seconds']:.4f}s"
)

print(
    "Interpretation: small jobs often dislike extreme over-partitioning, while larger shuffle jobs "
    "usually outperform when you avoid both under-parallelism and excessive tiny tasks."
)


## Experiment 3 — Skew Simulation

Goal: simulate skew where **99%** of alerts belong to one `endpoint_id`.

Why this matters:

A single hot key can make one partition do almost all the work. That creates:

- one long-running task
- executor imbalance
- poor end-to-end stage time even if the rest of the cluster is idle

We will:

1. Build a skewed alerts DataFrame
2. Join it to endpoints and measure runtime
3. Inspect partition size distribution
4. Apply **salting**
5. Compare runtimes before and after salting


In [ ]:
from pyspark.sql import Row

base_alerts = alerts_df.select("alert_id", "endpoint_id", "severity", "message", "created_at")
total_alerts = base_alerts.count()
hot_count = int(total_alerts * 0.99)
rest_count = total_alerts - hot_count
hot_endpoint_id = 1
salt_buckets = 16

hot_alerts = (
    base_alerts.orderBy("alert_id")
    .limit(hot_count)
    .withColumn("endpoint_id", F.lit(hot_endpoint_id))
)

rest_alerts = (
    base_alerts.orderBy(F.desc("alert_id"))
    .limit(rest_count)
    .withColumn("endpoint_id", ((F.col("alert_id") % 9999) + 2).cast("int"))
)

skewed_alerts_df = hot_alerts.unionByName(rest_alerts).repartition(200, "endpoint_id").cache()
_ = skewed_alerts_df.count()

print("Skewed rows:", skewed_alerts_df.count())
print("Hot endpoint share:", skewed_alerts_df.filter(F.col("endpoint_id") == hot_endpoint_id).count() / total_alerts)

def partition_distribution(df, label):
    counts = (
        df.rdd.mapPartitions(lambda it: [sum(1 for _ in it)])
        .zipWithIndex()
        .map(lambda x: (int(x[1]), int(x[0])))
        .toDF(["partition_id", "row_count"])
        .orderBy(F.desc("row_count"))
    )
    print(f"\nPartition distribution for {label}:")
    counts.show(20, truncate=False)
    summary = counts.agg(
        F.min("row_count").alias("min_rows"),
        F.max("row_count").alias("max_rows"),
        F.avg("row_count").alias("avg_rows")
    ).collect()[0]
    print(
        f"{label} summary -> min={summary['min_rows']}, "
        f"max={summary['max_rows']}, avg={summary['avg_rows']:.2f}"
    )
    return counts

def run_skewed_join():
    joined = skewed_alerts_df.join(endpoints_df, on="endpoint_id", how="inner")
    return joined.count()

_, skewed_join_time = timed_call("Skewed join without salting", run_skewed_join)

partition_distribution(
    skewed_alerts_df.select("alert_id", "endpoint_id").repartition(200, "endpoint_id"),
    "skewed alerts repartitioned by endpoint_id"
)

salted_alerts_df = (
    skewed_alerts_df.withColumn(
        "salt",
        F.when(F.col("endpoint_id") == hot_endpoint_id, F.pmod(F.col("alert_id"), F.lit(salt_buckets))).otherwise(F.lit(0))
    )
)

hot_endpoint_rows = endpoints_df.filter(F.col("endpoint_id") == hot_endpoint_id)
other_endpoint_rows = endpoints_df.filter(F.col("endpoint_id") != hot_endpoint_id).withColumn("salt", F.lit(0))

salt_values = spark.range(0, salt_buckets).select(F.col("id").cast("int").alias("salt"))
hot_endpoint_salted = hot_endpoint_rows.crossJoin(salt_values)

salted_endpoints_df = other_endpoint_rows.unionByName(hot_endpoint_salted).cache()
_ = salted_endpoints_df.count()

def run_salted_join():
    joined = salted_alerts_df.join(salted_endpoints_df, on=["endpoint_id", "salt"], how="inner")
    return joined.count()

_, salted_join_time = timed_call("Skewed join with salting", run_salted_join)

speedup = (skewed_join_time / salted_join_time) if salted_join_time > 0 else float("inf")
print(f"Skewed join speedup after salting: {speedup:.2f}x")

partition_distribution(
    salted_alerts_df.select("alert_id", "endpoint_id", "salt").repartition(200, "endpoint_id", "salt"),
    "salted alerts repartitioned by endpoint_id + salt"
)


## Experiment 4 — `coalesce()` vs `repartition()`

Goal: start with 200 partitions, then reduce to 4 partitions using:

- `coalesce(4)`
- `repartition(4)`

Why this matters:

- `coalesce()` tries to reduce partitions **without a full shuffle**
- `repartition()` performs a **full reshuffle** to rebalance data

Rule of thumb:

- use **coalesce** when shrinking partitions for output and you do **not** need perfect rebalance
- use **repartition** when you need more even partition distribution

We will compare write times to Parquet.


In [ ]:
import shutil
from pathlib import Path

base_output_dir = Path.cwd() / "spark_perf_lab_outputs"
base_output_dir.mkdir(parents=True, exist_ok=True)

write_source_df = metrics_df.repartition(200, "endpoint_id").cache()
_ = write_source_df.count()

print("Starting partitions:", write_source_df.rdd.getNumPartitions())

coalesce_path = str(base_output_dir / "coalesce_4")
repartition_path = str(base_output_dir / "repartition_4")

for path in [coalesce_path, repartition_path]:
    shutil.rmtree(path, ignore_errors=True)

def write_with_coalesce():
    df = write_source_df.coalesce(4)
    df.write.mode("overwrite").parquet(coalesce_path)
    return df.rdd.getNumPartitions()

def write_with_repartition():
    df = write_source_df.repartition(4)
    df.write.mode("overwrite").parquet(repartition_path)
    return df.rdd.getNumPartitions()

coalesce_parts, coalesce_time = timed_call("Write with coalesce(4)", write_with_coalesce)
repartition_parts, repartition_time = timed_call("Write with repartition(4)", write_with_repartition)

print(f"coalesce partitions seen before write: {coalesce_parts}")
print(f"repartition partitions seen before write: {repartition_parts}")
print(
    "Guidance: coalesce is usually cheaper when shrinking files late in the pipeline. "
    "Repartition is better when you need even task balance before expensive downstream work."
)


## Experiment 5 — Predicate Pushdown via JDBC

Goal: compare:

1. loading the full `metrics` table
2. loading only rows where `metric_name = 'cpu_utilization'`

Why this matters:

If the filter is applied in PostgreSQL first, Spark reads fewer rows over JDBC. That reduces:

- network transfer
- deserialization
- Spark-side scan work

This is one of the cheapest and most effective tuning moves in data pipelines.


In [ ]:
def load_full_metrics():
    df = read_jdbc_table(
        "(select endpoint_id, metric_name, value, timestamp from public.metrics) as metrics_full",
        num_partitions=16,
        partition_column="endpoint_id",
        lower_bound=1,
        upper_bound=10000
    )
    return df, df.count()

def load_cpu_metrics():
    df = read_jdbc_table(
        "(select endpoint_id, metric_name, value, timestamp from public.metrics where metric_name = 'cpu_utilization') as metrics_cpu",
        num_partitions=16,
        partition_column="endpoint_id",
        lower_bound=1,
        upper_bound=10000
    )
    return df, df.count()

(full_df, full_count), full_time = timed_call("Load full metrics table", load_full_metrics)
(cpu_df, cpu_count), cpu_time = timed_call("Load pushdown cpu_utilization only", load_cpu_metrics)

print(f"Full metrics rows: {full_count}")
print(f"Pushed-down cpu_utilization rows: {cpu_count}")

reduction = (1 - (cpu_count / full_count)) * 100 if full_count else 0.0
speedup = (full_time / cpu_time) if cpu_time > 0 else float("inf")

print(f"Row reduction from pushdown: {reduction:.2f}%")
print(f"Pushdown speedup: {speedup:.2f}x")

print("\nExplain plan for pushed-down DataFrame:")
cpu_df.explain(mode="formatted")


## What Just Happened

Here is the decision guide from the five experiments:

### 1) Broadcast join
Use broadcast when one side is clearly small enough to fit comfortably in memory.

**Good fit here:** `endpoints` (10K rows)  
**Benefit:** avoids a heavier shuffle join

### 2) Shuffle partition tuning
There is always a runtime sweet spot.

- too low → big slow tasks
- too high → too many tiny tasks and scheduler overhead

Tune this based on actual workload shape, not folklore.

### 3) Skew handling
One hot key can dominate a stage.

Symptoms:

- one or a few straggler tasks
- long tail stage time
- poor cluster utilization

Salting is a practical technique when a small set of keys creates severe imbalance.

### 4) Coalesce vs repartition
- **coalesce**: cheaper when shrinking output partitions late
- **repartition**: better when you need even distribution and are willing to pay for the shuffle

### 5) Predicate pushdown
Push filters into the source whenever possible.

This reduces:

- bytes read
- network movement
- Spark-side compute

## Bottom line

At Citi scale — **500K metrics, 25K alerts, 10K endpoints** — these tuning choices are the difference between a **30-second job** and a **5-minute job**.

The winning sequence is usually:

1. push down filters
2. broadcast small dimensions
3. fix skew
4. tune shuffle partitions
5. choose coalesce vs repartition intentionally


In [ ]:
summary_rows = [
    ("Broadcast join", f"{no_broadcast_time:.4f}s", f"{with_broadcast_time:.4f}s", f"{(no_broadcast_time / with_broadcast_time) if with_broadcast_time else float('inf'):.2f}x"),
    ("Partition tuning best", "-", f"{best_row['elapsed_seconds']:.4f}s @ {best_row['shuffle_partitions']} partitions", "-"),
    ("Skew salting", f"{skewed_join_time:.4f}s", f"{salted_join_time:.4f}s", f"{(skewed_join_time / salted_join_time) if salted_join_time else float('inf'):.2f}x"),
    ("Predicate pushdown", f"{full_time:.4f}s ({full_count} rows)", f"{cpu_time:.4f}s ({cpu_count} rows)", f"{(full_time / cpu_time) if cpu_time else float('inf'):.2f}x"),
]

summary_df = spark.createDataFrame(summary_rows, ["experiment", "baseline", "optimized", "speedup"])
summary_df.show(truncate=False)

print("Lab complete.")
